# 01 — Private preparation and baseline
Inputs: supplied CSV, private multilingual seeds, Python 3.12 and pinned dependencies.
Cap: 15 minutes after setup. Output: data audit, split manifest and immediately loadable baseline.
No final evaluation here. All examples remain unreviewed. Existing outputs are resumed, not overwritten.
For Colab, mount Drive with `from google.colab import drive; drive.mount('/content/drive')`,
copy the private project into `/content/svo`, and install `training/requirements-runtime.txt`.
Use local disk for compute; Drive is an optional checkpoint destination. Verify time remaining before running.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json
# Local: run from repo root or training/notebooks. Colab: copy private project to /content/svo.
ROOT = Path('/content/svo') if Path('/content/svo/training').exists() else Path.cwd()
while not (ROOT / 'training').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'training/src/svo').exists(), 'Set ROOT to the extracted project root'
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT / 'training/src')
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '2'
PYTHON = str(ROOT / 'training/.venv/bin/python') if (ROOT / 'training/.venv/bin/python').exists() else sys.executable
DATA = ROOT / 'training/data/processed/messages.jsonl'
DRIVE_ROOT = Path('/content/drive/MyDrive/ScamVariantObservatory')
def run(*args):
    subprocess.run([PYTHON, *map(str,args)], check=True, cwd=ROOT, env=os.environ.copy())
run('-c', 'import numpy, sklearn, joblib; print("Core dependencies ready")')


In [ ]:
if not DATA.exists():
    run('training/scripts/prepare_data.py', '--csv', 'training/final_dataset_output.csv',
        '--seeds', 'training/data/private/seeds.json', '--allow-unverified-source', '--output', DATA)
RUN = ROOT / 'training/runs/baseline'
if not (RUN / 'freeze.json').exists():
    run('-m', 'svo.train', '--data', DATA, '--run', RUN, '--simulation')
run('training/scripts/verify_bundle.py', RUN / 'bundle')
print(json.loads((RUN / 'development.json').read_text()))


In [ ]:
# Optional Colab checkpoint. Mount Drive before setting this flag.
COPY_TO_DRIVE = False
if COPY_TO_DRIVE:
    import shutil
    assert DRIVE_ROOT.parent.exists(), 'Mount Drive first'
    destination = DRIVE_ROOT / 'training-checkpoints' / RUN.name
    shutil.copytree(RUN, destination, dirs_exist_ok=False)
    print('Checkpoint copied; treat it as private.')
print('Local stage artifacts:', RUN)
